# 4.4 ops-sparse SpMV

## 本节学习目标

- 使用 CSR descriptor 调用稀疏库
- 区分 handle/descriptor 与 ACLNN executor

## 环境检查与 ops-sparse 准备

SpMV 除 CANN Toolkit 外还依赖独立的 `ops-sparse` 头文件和动态库。CMake 会检查 `OPS_SPARSE_ROOT` 和 `ASCEND_HOME`；缺少组件时，先按 `EXPERIMENT_GUIDE.md` 的安装步骤为 A3 构建并安装 `spmv` 包。

In [ ]:
%%bash
set -euo pipefail
command -v cmake
command -v npu-smi
npu-smi info
printf "ASCEND_HOME_PATH=%s\n" "${ASCEND_HOME_PATH:?请先 source CANN set_env.sh}"
found=0
for root in "${OPS_SPARSE_ROOT:-}" "${ASCEND_HOME:-${ASCEND_HOME_PATH}}"; do
  [[ -n "$root" ]] || continue
  if [[ -f "$root/include/cann_ops_sparse.h" && -f "$root/lib64/libops_sparse.so" ]]; then
    echo "ops-sparse found: $root"
    found=1
    break
  fi
done
[[ "$found" -eq 1 ]] || { echo '未找到 ops-sparse；请先按 EXPERIMENT_GUIDE.md 安装与当前 CANN 配套的组件。' >&2; exit 1; }


## 稀疏对象

工程创建 `aclsparseHandle_t` 并绑定 stream，使用 `aclsparseCreateCsr` 描述 CSR，使用 `aclsparseCreateDnVec` 描述 x/y。

## 执行与验证

`aclsparseSpMV` 传入 alpha、beta、operation、matrix、vectors 和数据类型；同步后回读 y，并与 Host CSR reference 比较相对误差。当前接口没有 ACLNN executor/workspace。

## 构建并运行

从当前章节目录执行下面的 Cell，并对照随后给出的检查点阅读输出。

In [ ]:
!cd src/acl_operator_calls/SpMV-acl && bash scripts/build.sh
!cd src/acl_operator_calls/SpMV-acl && bash scripts/run.sh --rows 100000 --cols 100000 --nnz 1000000 --warmup 10 --repeat 100

## 预期现象与结果分析

真实环境应输出 rows/cols/nnz、ACL SpMV 时间和误差。若 CMake 报缺少 `cann_ops_sparse.h` 或 `libops_sparse.so`，按 `EXPERIMENT_GUIDE.md` 安装配套组件，并用 `OPS_SPARSE_ROOT` 指定安装位置。

## 原工程历史参考输出

`acl-c/README.md` 记录了 CANN 9.0.0 下 `100000×100000`、`1000000 nnz`、`warmup=10`、`repeat=100` 的远程 Ascend 实测：平均 `76.679907 ms`，相对误差 `7.581571451e-08`，低于源码的 `1e-6` 门槛。这里首先用它核对 `aclsparseSpMV` 结果正确；性能数字只代表当时的软件栈和输入。

## 课后实践

画出 handle、CSR descriptor、dense vector descriptor 与 Device buffer 的关系。

参考答案见 `answer/04.04_answer.md`。

## 直接执行实验

该 Cell 强制真实 ACL/ops-sparse 后端并独立运行 SpMV；记录 Device ID、设备时间和 CPU reference 误差。


In [ ]:
%%bash
set -e
cd src/acl_operator_calls/SpMV-acl
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DACL_C_STUB=OFF
cmake --build build -j
./build/bin/spmv_acl --rows 4096 --cols 4096 --nnz 65536 --warmup 3 --repeat 10 --device 0
